# 04 — Next-Week Blood Demand Regression
**CMD:** `jupyter lab` → **Run All**; alternative command: `python -m src.prediction --train-regression`.

**Purpose:** compare three regression models for next-week demand using MAE, RMSE, and MAPE; choose by validation MAE.

In [ ]:
# CMD / purpose: Run this cell first in each notebook.
# Makes imports work whether Jupyter starts from the project root or notebooks/.
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open Jupyter from the blood-shortage-ml project directory.')
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

In [ ]:
# CMD / purpose: train regressors and save best_regressor.joblib, regression_metrics.csv.
from src.prediction import train_regression
from src.feature_engineering import load_processed
results = train_regression(load_processed())
display(results.round(3))

In [ ]:
# CMD / purpose: plot held-out test actual next-week demand against predicted demand.
import joblib
import matplotlib.pyplot as plt
import numpy as np
from src.prediction import REGRESSOR_PATH, chronological_splits
from src.feature_engineering import FEATURE_COLUMNS
df = load_processed().dropna(subset=['next_demand','target_date'])
test = chronological_splits(df)['test']
bundle = joblib.load(REGRESSOR_PATH)
predicted = np.clip(bundle['pipeline'].predict(test[FEATURE_COLUMNS]), 0, None)
actual = test.next_demand.to_numpy()
plt.figure(figsize=(6,5))
plt.scatter(actual, predicted, s=12, alpha=.45)
lim = max(actual.max(), predicted.max())
plt.plot([0,lim],[0,lim],linestyle='--',color='grey')
plt.xlabel('Observed next-week demand');plt.ylabel('Predicted next-week demand')
plt.title(f"Test forecasts: {bundle['model_name']}")
plt.tight_layout();plt.show()

In [ ]:
# CMD / purpose: report error for each blood type, not only combined error.
import pandas as pd
from sklearn.metrics import mean_absolute_error
output = test[['blood_type', 'next_demand']].copy()
output['predicted_demand'] = predicted
display(output.groupby('blood_type').apply(lambda g: pd.Series({'MAE':mean_absolute_error(g.next_demand,g.predicted_demand),'test_rows':len(g)}),include_groups=False).round(2))

**Interpretation:** MAE is the average absolute prediction error measured in demand units. Test MAPE excludes cases where actual demand is zero. A demand forecast is not the same as a forecast of available inventory.